# S04 — GT-free spatial evidence and oracle gap

This notebook executes deterministic constructed cases at the sender/frame/deadline decision unit. The operational path uses only receiver-owned fixed tiles and receiver context. Evaluator boxes and labels are joined only after both tracks have frozen decisions. `T` is conformity, not an honesty probability. No detector, policy, external data, or later stage is used.

In [ ]:
import json, os, platform, sys
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
from matplotlib.figure import Figure
from IPython.display import Markdown, display
from valid_point.evaluation.gap import run_spatial_experiment
from valid_point.provenance import export_table, sha256_file, write_json_new
root = Path(os.environ['VP_REPO_ROOT']).resolve()
run = Path(os.environ['VP_RUN_DIR']).resolve()
config_path = root / 'configs/spatial_tracks.json'
config = json.loads(config_path.read_text())
provenance = {'run_id': run.name, 'stage': 'S04', 'track': 'paired', 'segment': 'constructed_gap_cases', 'config_sha256': sha256_file(config_path), 'source_sha256': os.environ['VP_SOURCE_SHA256'], 'notebook_section': 'NB04'}
inventory = {'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'matplotlib': matplotlib.__version__}
display(Markdown('## Frozen inputs, units, deadline, config, and seeds'))
print(json.dumps({'environment': inventory, 'decision': config['decision'], 'units': {'geometry': 'm', 'raw spatial measurement': 'points', 'time': 'ns', 'T': 'dimensionless conformity'}, 'grid': config['gt_free_grid'], 'receiver_context': config['receiver_context'], 'splits': config['splits'], 'optional_ego_proposals': config['optional_ego_proposals']}, indent=2))

## Track separation

The GT-free grid is frozen at 0 ns, before the 1.5 s anchor and before every tested cloud. Missing trusted range uses the preregistered broad receiver context; no sender range claim is accepted. Oracle boxes are an evaluator-only research exception. Each track has disjoint reference and calibration seeds and fits its own `u`, `b`, and clean threshold.

In [ ]:
result = run_spatial_experiment(config)
display(Markdown('## Geometry, independently fitted references, and calibrations'))
print(json.dumps({'grid': result['grid'], 'references': result['references'], 'calibrations': result['calibrations']}, indent=2))
assert result['grid']['tile_count'] == config['gt_free_grid']['required_tile_count']
assert result['references']['gt_free']['reference_seeds'] != result['references']['oracle_box']['reference_seeds']
assert result['calibrations']['gt_free']['calibration_seeds'] != result['calibrations']['oracle_box']['calibration_seeds']

In [ ]:
display(Markdown('## Raw measurements, unknown states, outcomes, and failures'))
columns = ['case','raw_point_count','gt_free_max_tile_count','gt_free_nonempty_tiles','gt_free_outside_roi_points','gt_free_context','oracle_box_counts','gt_free_status','gt_free_T','gt_free_alarm','oracle_status','oracle_T','oracle_alarm','gt_free_outcome','oracle_outcome','proposal_state_deferred','association_state','visibility','object_count_evaluator_only']
for row in result['cases']:
    print(json.dumps({key: row[key] for key in columns}, sort_keys=True))
display(Markdown('Unknown visibility is retained for every case. Present-empty is a known zero count and is not accused; absent-cloud is unknown and abstains. Object count is displayed only in the evaluator column and never enters GT-free scoring.'))

In [ ]:
display(Markdown('## T04_gap_cases and F04_oracle_gap'))
table_rows = []
for row in result['cases']:
    table_rows.append({key: ('unknown' if value is None else value) for key, value in row.items()})
table_paths = export_table(table_rows, 'T04_gap_cases', root / 'reports/tables', provenance)
labels = [row['case'] for row in result['cases']]
gt_values = [np.nan if row['gt_free_T'] is None else row['gt_free_T'] for row in result['cases']]
oracle_values = [np.nan if row['oracle_T'] is None else row['oracle_T'] for row in result['cases']]
x = np.arange(len(labels))
fig = Figure(figsize=(15, 6.5), dpi=150)
ax = fig.subplots()
ax.bar(x-.2, gt_values, width=.4, label='GT-free fixed tiles T', color='#377eb8')
ax.bar(x+.2, oracle_values, width=.4, label='Oracle boxes T', color='#e41a1c', alpha=.78)
for index, row in enumerate(result['cases']):
    if row['gt_free_alarm'] is True: ax.scatter(index-.2, 1.05, marker='v', color='#08306b', s=35)
    if row['oracle_alarm'] is True: ax.scatter(index+.2, 1.05, marker='v', color='#99000d', s=35)
    if row['gt_free_status'] != 'known': ax.text(index-.2, .5, 'UNKNOWN', rotation=90, ha='center', va='center', fontsize=7)
ax.set_ylim(0, 1.12); ax.set_ylabel('T (dimensionless conformity; lower is more anomalous)')
ax.set_xticks(x, labels, rotation=42, ha='right'); ax.legend(ncol=2, loc='upper center')
ax.set_title('F04_oracle_gap — paired decisions; triangles mark frozen-threshold alarms')
ax.grid(axis='y', alpha=.2)
fig.tight_layout()
stem = '_'.join(('F04_oracle_gap', provenance['stage'], provenance['track'], provenance['segment'], provenance['run_id']))
figure_paths = [root / 'reports/figures' / f'{stem}.{ext}' for ext in ('pdf','svg','png')]
for path in figure_paths: path.parent.mkdir(parents=True, exist_ok=True); fig.savefig(path, bbox_inches='tight')
write_json_new(root / 'reports/figures' / f'{stem}.provenance.json', {**provenance, 'name': 'F04_oracle_gap', 'outputs': {path.name: sha256_file(path) for path in figure_paths}})
write_json_new(run / 'spatial_gap_result.json', result)
write_json_new(run / 'reference_lineage.json', result['references'])
write_json_new(run / 'calibration_lineage.json', result['calibrations'])
print('tables:', [str(path.relative_to(root)) for path in table_paths])
print('figures:', [str(path.relative_to(root)) for path in figure_paths])
print(json.dumps(result['gap'], indent=2))

In [ ]:
display(Markdown('## Contract checks and visible gate'))
print(json.dumps(result['contract_checks'], indent=2))
assert all(result['contract_checks'].values())
display(Markdown('### Technical PASS; G3 FAIL — explicit negative result'))
print(json.dumps(result['gate'], indent=2))
display(Markdown('The operational fixed-tile path is isolated from GT and reproducible, but these constructed cases do not make the GT-free score defensible for operational readiness. It misses split-across-tile, outside-ROI, and count-preserving cases; it false-alarms on a unique honest view and changing benign content; and it abstains when the cloud is absent. Fixed tiles do expose concentrated in-ROI addition and an empty-region ghost. Therefore real-policy and operational-readiness claims remain blocked.'))